# 02 - TF-IDF + Rocchio Feedback

Notebook này xây dựng phương pháp truy xuất **TF-IDF + Rocchio Feedback**.

Luồng chính:

```text
news_processed.pkl
→ processed_text
→ TF-IDF thủ công
→ Cosine Similarity
→ Rocchio Feedback
→ Search lại
→ Top-k kết quả
```

## Cell 1 - Import thư viện

Cell này import các thư viện cần thiết để:
- Đọc dữ liệu đã xử lý.
- Tự tính TF-IDF.
- Tính cosine similarity.
- Lưu model/index ra folder `models/tfidf_rocchio`.

In [ ]:
import os
import math
import pickle
import pandas as pd
import numpy as np

from collections import Counter
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

## Cell 2 - Import hàm xử lý query dùng chung

Cả 3 phương pháp nên dùng chung hàm xử lý query trong `src/preprocess.py` để đảm bảo query được xử lý giống document.

In [ ]:
import sys

CURRENT_DIR = os.getcwd()

if os.path.basename(CURRENT_DIR) == "notebooks":
    PROJECT_ROOT = os.path.abspath("..")
else:
    PROJECT_ROOT = CURRENT_DIR

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.preprocess import preprocess_query

## Cell 3 - Khai báo đường dẫn

Cell này khai báo:
- File dữ liệu đã xử lý.
- Folder lưu index/model của phương pháp TF-IDF + Rocchio.

In [ ]:
DATA_PATH = "../data/processed/news_processed.pkl"

MODEL_DIR = "../models/tfidf_rocchio"
os.makedirs(MODEL_DIR, exist_ok=True)

## Cell 4 - Load dữ liệu đã xử lý

Dữ liệu đầu vào là file `news_processed.pkl` được tạo từ notebook preprocessing chung của nhóm.

In [ ]:
df = pd.read_pickle(DATA_PATH)

print("Shape:", df.shape)
print(df.columns.tolist())

df[["doc_id", "title", "topic", "source", "processed_text"]].head()

## Cell 5 - Lấy danh sách document

Phương pháp TF-IDF sử dụng cột `processed_text` làm nội dung truy xuất.

In [ ]:
documents = df["processed_text"].fillna("").astype(str).tolist()

print("Số lượng documents:", len(documents))
print(documents[0][:500])

## Cell 6 - Tách token từ document

Vì `processed_text` đã được tách từ ở bước preprocessing, ở đây chỉ cần dùng `.split()`.

In [ ]:
tokenized_docs = [doc.split() for doc in documents]

print(tokenized_docs[0][:50])

## Cell 7 - Tạo vocabulary

Cell này xây dựng vocabulary bằng cách:
- Đếm số document chứa mỗi token.
- Loại token quá hiếm bằng `min_df`.
- Loại token quá phổ biến bằng `max_df_ratio`.
- Không giới hạn `max_features`, lấy toàn bộ token hợp lệ sau lọc.

In [ ]:
min_df = 5
max_df_ratio = 0.85

N = len(tokenized_docs)

df_counter = Counter()

for tokens in tokenized_docs:
    df_counter.update(set(tokens))

max_df = int(max_df_ratio * N)

valid_tokens = {
    token: df_count
    for token, df_count in df_counter.items()
    if df_count >= min_df and df_count <= max_df
}

valid_tokens_sorted = sorted(
    valid_tokens.items(),
    key=lambda x: x[1],
    reverse=True
)

vocab = {
    token: idx
    for idx, (token, _) in enumerate(valid_tokens_sorted)
}

print("Số documents:", N)
print("Tổng số token ban đầu:", len(df_counter))
print("Vocabulary size sau lọc:", len(vocab))

## Cell 8 - Tính IDF thủ công

IDF được tính theo công thức có làm trơn:

```text
IDF(t) = log((N + 1) / (DF(t) + 1)) + 1
```

In [ ]:
idf = np.zeros(len(vocab), dtype=np.float32)

for token, idx in vocab.items():
    df_t = df_counter[token]
    idf[idx] = math.log((N + 1) / (df_t + 1)) + 1

print("IDF shape:", idf.shape)
print(idf[:10])

## Cell 9 - Tính TF-IDF Matrix thủ công

Cell này tự tính ma trận TF-IDF, không dùng `TfidfVectorizer`.

Công thức TF dùng dạng sublinear:

```text
TF(t, d) = 1 + log(count(t, d))
TF-IDF(t, d) = TF(t, d) × IDF(t)
```

In [ ]:
rows = []
cols = []
values = []

for doc_idx, tokens in enumerate(tokenized_docs):
    token_counts = Counter(tokens)

    for token, count in token_counts.items():
        if token in vocab:
            col_idx = vocab[token]

            tf = 1 + math.log(count)
            tfidf_value = tf * idf[col_idx]

            rows.append(doc_idx)
            cols.append(col_idx)
            values.append(tfidf_value)

tfidf_matrix = csr_matrix(
    (values, (rows, cols)),
    shape=(N, len(vocab)),
    dtype=np.float32
)

tfidf_matrix = normalize(tfidf_matrix, norm="l2", axis=1)

print("TF-IDF Matrix:", tfidf_matrix.shape)
print("Non-zero values:", tfidf_matrix.nnz)

## Cell 10 - Tạo metadata

Metadata dùng để hiển thị kết quả truy xuất sau khi tìm được `doc_id` / index của document.

In [ ]:
metadata = df[[
    "doc_id",
    "id",
    "title",
    "content",
    "topic",
    "source",
    "url"
]].copy()

metadata.head()

## Cell 11 - Lưu index/model

Cell này lưu các thành phần cần dùng lại khi search:
- `vocab.pkl`: từ điển token → cột.
- `idf.pkl`: trọng số IDF.
- `tfidf_matrix.pkl`: ma trận TF-IDF của document.
- `metadata.pkl`: thông tin bài báo để hiển thị.

In [ ]:
with open(f"{MODEL_DIR}/vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

with open(f"{MODEL_DIR}/idf.pkl", "wb") as f:
    pickle.dump(idf, f)

with open(f"{MODEL_DIR}/tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)

with open(f"{MODEL_DIR}/metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

print("Đã lưu model/index vào:", MODEL_DIR)

## Cell 12 - Chuyển query sang TF-IDF

Query phải được xử lý giống document, sau đó chuyển thành vector TF-IDF theo cùng `vocab` và `idf`.

In [ ]:
def query_to_tfidf(query_tokens):
    token_counts = Counter(query_tokens)

    rows = []
    cols = []
    values = []

    for token, count in token_counts.items():
        if token in vocab:
            col_idx = vocab[token]

            tf = 1 + math.log(count)
            tfidf_value = tf * idf[col_idx]

            rows.append(0)
            cols.append(col_idx)
            values.append(tfidf_value)

    query_vector = csr_matrix(
        (values, (rows, cols)),
        shape=(1, len(vocab)),
        dtype=np.float32
    )

    query_vector = normalize(query_vector, norm="l2", axis=1)

    return query_vector

## Cell 13 - Hàm search TF-IDF cơ bản

Hàm này:
1. Xử lý query.
2. Chuyển query sang TF-IDF.
3. Tính cosine similarity với toàn bộ document.
4. Trả về top-k kết quả.

In [ ]:
def search_tfidf(query, top_k=10):
    query_processed, query_tokens = preprocess_query(query)

    query_vector = query_to_tfidf(query_tokens)

    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

    top_indices = scores.argsort()[::-1][:top_k]

    results = metadata.iloc[top_indices].copy()
    results["score"] = scores[top_indices]
    results["query_processed"] = query_processed

    return results

## Cell 14 - Hàm Rocchio Feedback

Rocchio cập nhật vector query dựa trên:
- Các document liên quan.
- Các document không liên quan.

Công thức ý tưởng:

```text
q_new = alpha * q_old + beta * mean(relevant_docs) - gamma * mean(non_relevant_docs)
```

In [ ]:
def rocchio_feedback(
    query_vector,
    relevant_indices=None,
    non_relevant_indices=None,
    alpha=1.0,
    beta=0.75,
    gamma=0.15
):
    relevant_indices = relevant_indices or []
    non_relevant_indices = non_relevant_indices or []

    q_new = alpha * query_vector

    if len(relevant_indices) > 0:
        rel_centroid = tfidf_matrix[relevant_indices].mean(axis=0)
        rel_centroid = csr_matrix(rel_centroid)
        q_new = q_new + beta * rel_centroid

    if len(non_relevant_indices) > 0:
        nonrel_centroid = tfidf_matrix[non_relevant_indices].mean(axis=0)
        nonrel_centroid = csr_matrix(nonrel_centroid)
        q_new = q_new - gamma * nonrel_centroid

    q_new = normalize(q_new, norm="l2", axis=1)

    return q_new

## Cell 15 - Hàm search TF-IDF + Rocchio

Hàm này search lần đầu, sau đó dùng một số kết quả đầu làm pseudo feedback:
- Top `relevant_top_n` kết quả đầu được xem là liên quan.
- Một số kết quả cuối trong top-k được xem là không liên quan.

Đây là dạng **pseudo relevance feedback**, không cần người dùng gán nhãn thủ công.

In [ ]:
def search_tfidf_rocchio(
    query,
    top_k=10,
    feedback_pool=20,
    relevant_top_n=5,
    non_relevant_bottom_n=5,
    alpha=1.0,
    beta=0.75,
    gamma=0.15
):
    query_processed, query_tokens = preprocess_query(query)

    query_vector = query_to_tfidf(query_tokens)

    initial_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    initial_indices = initial_scores.argsort()[::-1][:feedback_pool]

    relevant_indices = list(initial_indices[:relevant_top_n])
    non_relevant_indices = list(initial_indices[-non_relevant_bottom_n:])

    updated_query_vector = rocchio_feedback(
        query_vector=query_vector,
        relevant_indices=relevant_indices,
        non_relevant_indices=non_relevant_indices,
        alpha=alpha,
        beta=beta,
        gamma=gamma
    )

    scores = cosine_similarity(updated_query_vector, tfidf_matrix).flatten()
    top_indices = scores.argsort()[::-1][:top_k]

    results = metadata.iloc[top_indices].copy()
    results["score"] = scores[top_indices]
    results["query_processed"] = query_processed

    return results

## Cell 16 - Test TF-IDF cơ bản

Cell này chạy thử phương pháp TF-IDF chưa dùng Rocchio.

In [ ]:
query = "cầu thủ Quang Hải"

results = search_tfidf(query, top_k=10)

print("Query gốc:", query)
print("Query sau xử lý:", results["query_processed"].iloc[0])

results[[
    "doc_id",
    "title",
    "topic",
    "source",
    "url",
    "score"
]]

## Cell 17 - Test TF-IDF + Rocchio

Cell này chạy thử phương pháp TF-IDF sau khi áp dụng Rocchio Feedback.

In [ ]:
query = "cầu thủ Quang Hải"

results = search_tfidf_rocchio(query, top_k=10)

print("Query gốc:", query)
print("Query sau xử lý:", results["query_processed"].iloc[0])

results[[
    "doc_id",
    "title",
    "topic",
    "source",
    "url",
    "score"
]]

## Cell 18 - Hiển thị kết quả dạng dễ đọc

Cell này in kết quả theo từng rank để dễ kiểm tra nội dung bài báo.

In [ ]:
def show_results(results):
    for rank, (_, row) in enumerate(results.iterrows(), start=1):
        print("=" * 100)
        print(f"Rank {rank}")
        print("Score :", round(row["score"], 4))
        print("Title :", row["title"])
        print("Topic :", row["topic"])
        print("Source:", row["source"])
        print("URL   :", row["url"])
        print("Content preview:")
        print(str(row["content"])[:500])

In [ ]:
show_results(results)